In [342]:
import pandapipes as pp
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
import numpy as np
from tespy.tools import UserDefinedEquation


class Bidirectional_W_to_WHeatPump:

    def __init__( self,name,net,refrigerant,HC_ext_id,HC_inj_id):
        self.name = name
        self.refrigerant = refrigerant
        self.HC_ext_id=HC_ext_id
        self.HC_inj_id=HC_inj_id
        self.net=net
        self._build_tespy_Cycles()

    def _build_tespy_Cycles(self):
        def my_ude(ude):
            dt_DHN=ude.params['dt']
            return ude.conns[0].calc_T()+dt_DHN-ude.conns[1].calc_T()
        def my_ude_dependents(ude):
            c1, c2 = ude.conns
            return [c1.p,c1.h, c2.p,c2.h]
        self.nw_Cooling_net = Network()
        self.nw_Cooling_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Cooling_net_compressor = Compressor("compresor")
        self.Cooling_net_condenser = Condenser("condensador")
        self.Cooling_net_valve = Valve("valvula_expansion")
        self.Cooling_net_evaporator = HeatExchanger("evaporador")
        self.Cooling_net_cc=CycleCloser('CycleCloser')
        self.Cooling_net_source_consumer=Source("Source_Consumer ")
        self.Cooling_net_sink_consumer=Sink("Sink_consumer")
        self.Cooling_net_source_reseau=Source("Source_Reseau")
        self.Cooling_net_sink_reseau=Sink("Sink_Reseau")
        self.Cooling_net_c0=Connection(self.Cooling_net_valve, 'out1', self.Cooling_net_cc, 'in1', label='0')
        self.Cooling_net_c1 = Connection(self.Cooling_net_cc, 'out1', self.Cooling_net_evaporator, 'in2', label='1')
        self.Cooling_net_c2 = Connection(self.Cooling_net_evaporator, 'out2', self.Cooling_net_compressor, 'in1', label='2')
        self.Cooling_net_c3 = Connection(self.Cooling_net_compressor, 'out1', self.Cooling_net_condenser, 'in1', label='3')
        self.Cooling_net_c4 = Connection(self.Cooling_net_condenser, 'out1', self.Cooling_net_valve, 'in1', label='4')
        self.Cooling_net_c5=Connection(self.Cooling_net_condenser, 'out2',self.Cooling_net_sink_consumer, 'in1', label='5')
        self.Cooling_net_c6=Connection(self.Cooling_net_source_consumer, 'out1',self.Cooling_net_condenser , 'in2', label='6')
        self.Cooling_net_c7=Connection(self.Cooling_net_source_reseau, 'out1',self.Cooling_net_evaporator , 'in1', label='7')
        self.Cooling_net_c8=Connection(self.Cooling_net_evaporator, 'out1',self.Cooling_net_sink_reseau , 'in1', label='8')
        self.nw_Cooling_net.add_conns(self.Cooling_net_c0,  self.Cooling_net_c1,  self.Cooling_net_c2,  self.Cooling_net_c3,  self.Cooling_net_c4 , self.Cooling_net_c5,  self.Cooling_net_c6, self.Cooling_net_c7, self.Cooling_net_c8)
        self.Cooling_ude = UserDefinedEquation(
                    'my_ude', my_ude, my_ude_dependents, conns=[self.Cooling_net_c8, self.Cooling_net_c7],params={'dt': 5})
        self.nw_Cooling_net.add_ude(self.Cooling_ude)
        
        #Heating net 
        self.nw_Heating_net = Network()
        self.nw_Heating_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Heating_net_compressor = Compressor("compresor")
        self.Heating_net_condenser = Condenser("condensador")
        self.Heating_net_valve = Valve("valvula_expansion")
        self.Heating_net_evaporator = HeatExchanger("evaporador")
        self.Heating_net_cc=CycleCloser('CycleCloser')
        self.Heating_net_source_consumer=Source("Source_Consumer ")
        self.Heating_net_sink_consumer=Sink("Sink_consumer")
        self.Heating_net_source_reseau=Source("Source_Reseau")
        self.Heating_net_sink_reseau=Sink("Sink_Reseau")
        self.Heating_net_c0=Connection(self.Heating_net_valve, 'out1', self.Heating_net_cc, 'in1', label='0')
        self.Heating_net_c1 = Connection(self.Heating_net_cc, 'out1', self.Heating_net_evaporator, 'in2', label='1')
        self.Heating_net_c2 = Connection(self.Heating_net_evaporator, 'out2', self.Heating_net_compressor, 'in1', label='2')
        self.Heating_net_c3 = Connection(self.Heating_net_compressor, 'out1', self.Heating_net_condenser, 'in1', label='3')
        self.Heating_net_c4 = Connection(self.Heating_net_condenser, 'out1', self.Heating_net_valve, 'in1', label='4')
        self.Heating_net_c5 = Connection(self.Heating_net_evaporator, 'out1', self.Heating_net_sink_consumer, 'in1', label='5')
        self.Heating_net_c6 = Connection(self.Heating_net_source_consumer, 'out1', self.Heating_net_evaporator, 'in1', label='6')  
        self.Heating_net_c7 = Connection(self.Heating_net_source_reseau, 'out1', self.Heating_net_condenser, 'in2', label='7')
        self.Heating_net_c8 = Connection(self.Heating_net_condenser, 'out2', self.Heating_net_sink_reseau, 'in1', label='8')
        self.nw_Heating_net.add_conns(self.Heating_net_c0,  self.Heating_net_c1,  self.Heating_net_c2,  self.Heating_net_c3,  self.Heating_net_c4 , self.Heating_net_c5,  self.Heating_net_c6, self.Heating_net_c7, self.Heating_net_c8)
        self.Heating_ude = UserDefinedEquation(
                        'my_ude_4', my_ude, my_ude_dependents, conns=[self.Heating_net_c7, self.Heating_net_c8],params={'dt': 5})
        self.nw_Heating_net.add_ude(self.Heating_ude)
        
    def solve_cycle(self,mode,Q_consumer,eta_s,T_nework_in,T_cons,dt_DHN):
        #The pressure values for the consumer and district heating side in the heating pump are arbitrary values
        self.mode=mode
        T_cons_in = T_cons[0]
        T_cons_out = T_cons[1]
        T_DH_in = T_nework_in
        if self.mode=="COOLING_NET":
            self.Cooling_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
            self.Cooling_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5,Q=Q_consumer)
            self.Cooling_net_compressor.set_attr(eta_s=eta_s)
            self.Cooling_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Cooling_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Cooling_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Cooling_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Cooling_net._set_p_range([p_triple, p_high])
            self.nw_Cooling_net._set_h_range([h_min,h_max])
            self.Cooling_ude.params['dt'] = dt_DHN
            self.nw_Cooling_net.solve('design')        
        elif self.mode=="HEATING_NET":
            self.Heating_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5,Q=Q_consumer)
            self.Heating_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
            self.Heating_net_compressor.set_attr(eta_s=eta_s)
            self.Heating_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Heating_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Heating_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Heating_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Heating_net._set_p_range([p_triple, p_high])
            self.nw_Heating_net._set_h_range([h_min,h_max])
            self.Heating_ude.params['dt'] = dt_DHN
            self.nw_Heating_net.solve('design')
    def interaction_simulation(self,dT_water):
        if self.mode=="COOLING_NET":
            self.net.heat_consumer.at[self.HC_ext_id, "qext_w"] =abs( self.Cooling_net_evaporator.Q.val)
            self.net.heat_consumer.at[self.HC_ext_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False
            
        elif self.mode=="HEATING_NET":
            self.net.heat_consumer.at[self.HC_inj_id, "qext_w"] = self.Heating_net_condenser.Q.val
            self.net.heat_consumer.at[self.HC_inj_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False

        else: 
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False

    def get_main_results(self):
        if self.mode=="COOLING_NET":
            results = {
                "Q_consumer": abs(self.Cooling_net_condenser.Q.val),
                "Q_network": abs(self.Cooling_net_evaporator.Q.val),
                "W_compressor":self.Cooling_net_compressor.P.val ,
                "COP": abs(self.Cooling_net_condenser.Q.val)/self.Cooling_net_compressor.P.val 
            }
        elif self.mode=="HEATING_NET":
            results = {
                "Q_consumer": abs(self.Heating_net_evaporator.Q.val),
                "Q_network": abs(self.Heating_net_condenser.Q.val),
                "COP": abs(self.Heating_net_evaporator.Q.val)/self.Heating_net_compressor.P.val 
            }
        else:
            results = {}
        return results

    def get_results_solver(self):
        if self.mode=="COOLING_NET":
            self.nw_Cooling_net.print_results()
        elif self.mode=="HEATING_NET":
            self.nw_Heating_net.print_results()
        else:
            return None

In [343]:
import numpy as np
from pandapipes.component_models.abstract_models.circulation_pump import CirculationPump
from pandapipes.idx_branch import D, AREA, TOUTINIT
from pandapipes.idx_node import TINIT
from pandapipes.pf.pipeflow_setup import get_lookup

def _create_pit_branch_entries_fixed(cls, net, branch_pit):
    node_pit = net["_pit"]["node"]
    junction_idx_lookups = get_lookup(net, "node", "index")[
        cls.get_connected_node_type().table_name()]

    # FIX: usar la tabla COMPLETA, sin filtrar por in_service,
    # para que coincida con el tamaño de circ_pump_pit (que sí incluye
    # todas las filas, activas o no).
    circ_pump_tbl = net[cls.table_name()]
    circ_pump_pit = super(CirculationPump, cls).create_pit_branch_entries(net, branch_pit)
    circ_pump_pit[:, D] = 0.1
    circ_pump_pit[:, AREA] = circ_pump_pit[:, D] ** 2 * np.pi / 4

    types = circ_pump_tbl.type.values
    mask_t = np.isin(types, ["pt", "t"])
    juncts = circ_pump_tbl[cls.from_to_node_cols()[1]].values
    to_nodes = junction_idx_lookups[juncts]
    circ_pump_pit[mask_t, TOUTINIT] = circ_pump_tbl.t_flow_k.values[mask_t]
    circ_pump_pit[~mask_t, TOUTINIT] = node_pit[to_nodes[~mask_t], TINIT]
    return circ_pump_pit

CirculationPump.create_pit_branch_entries = classmethod(_create_pit_branch_entries_fixed)

In [344]:
net = pp.create_empty_network(fluid="water")
# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_fuente_inter_normal=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida") 
j_HP_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_Ida")
j_HP_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_Ida")
#Return
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_fuente_inter_retro=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_retro")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret") 
j_HP_ret_1= pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_ret")
j_HP_ret_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_ret")

#Plant-Nodo
u=0.35 / (np.pi * 0.15)
#Planta 1
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u, text_k=273.2, name="Tubo_Ida_Plant_Nodo_1",k_mm=0.1*1000
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_RetornoPlant_Nodo_1",k_mm=0.1*1000)

#Nodo-nodo
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="nodo_1_nodo_2",k_mm=0.1*1000
)
pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="nodo_2_ret_nodo_1_ret",k_mm=0.1*1000
)

#Node HP
#Node_Cons1

pipe_ida_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_1,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Ida_nodo_2_cons_1_ida",k_mm=0.1*1000
)
pipe_retorno_3= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_1, to_junction=j_nodo_2_ret,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Retorno_nodo_2_cons_1_ret",k_mm=0.1*1000
)
#Node_Cons2
pipe_ida_4 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_2,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Ida_nodo_2_cons_2_ida",k_mm=0.1*1000
)
pipe_retorno_4= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_2, to_junction=j_nodo_2_ret,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Retorno_nodo_2_cons_2_ret",k_mm=0.1*1000
)

#Heat Pumps
#Heat pump: extraction
HP_1_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_1 ,
    to_junction=j_HP_ret_1,
    qext_w=150000,
    deltat_k=50,
    name="HP_1_EXT"
)
#Heat mump 1: Injection
HP_1_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_1,
    to_junction=j_HP_1,
    qext_w=-150000,
    deltat_k=50,
    name="HP_1_INJ"
)
#Heat pump: extraction
HP_2_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_2 ,
    to_junction=j_HP_ret_2,
    qext_w=150000,
    deltat_k=50,
    name="HP_2_EXT"
)
#Heat mump 1: Injection
HP_2_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_2,
    to_junction=j_HP_2,
    qext_w=150000,
    deltat_k=-50,
    name="HP_2_INJ"
)
#Plant: 
Plant_1_normal=pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3,plift_bar=0.5,t_flow_k=340)
Plant_1_ret=pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ret,return_junction=j_fuente_ida,p_flow_bar=2.5,plift_bar=-0.5,t_flow_k=305)

T_network_inital_guess_1=35
T_network_inital_guess_2=50

In [345]:
HP_1=Bidirectional_W_to_WHeatPump(name="heat_pump_1",refrigerant="R134a",net=net,HC_ext_id=HP_1_Ext,HC_inj_id=HP_1_inj)
HP_2=Bidirectional_W_to_WHeatPump(name="heat_pump_2",refrigerant="R134a",net=net,HC_ext_id=HP_2_Ext,HC_inj_id=HP_2_inj)
T_network_HP_1=T_network_inital_guess_1
T_network_HP_2=T_network_inital_guess_2
T_network_HP_1_loop=T_network_inital_guess_1
T_network_HP_2_loop=T_network_inital_guess_2
tolerance=1E-6
error_1=10
error_2=10
iterations=0
while error_1>tolerance or error_2>tolerance:
    HP_1.solve_cycle(mode="COOLING_NET",Q_consumer=-150,eta_s=0.95,T_nework_in=T_network_HP_1,T_cons=[50,60],dt_DHN=5)
    HP_2.solve_cycle(mode="HEATING_NET",Q_consumer=-150000,eta_s=0.95,T_nework_in=T_network_HP_2,T_cons=[35,25],dt_DHN=10)
    print("iteration")
    HP_1.interaction_simulation(dT_water=5)
    HP_2.interaction_simulation(dT_water=-10)
    try:
        net.circ_pump_pressure.at[Plant_1_normal, "in_service"] = True
        net.circ_pump_pressure.at[Plant_1_ret, "in_service"] = False
        pp.pipeflow(net, mode="bidirectional")
    except UserWarning as e:
        print("modo directo falla:", e)
        net.circ_pump_pressure.at[Plant_1_normal, "in_service"] = False
        net.circ_pump_pressure.at[Plant_1_ret, "in_service"] = True
        pp.pipeflow(net, mode="bidirectional")
    T_network_HP_1=float(net.res_heat_consumer.at[HP_1.HC_ext_id,"t_from_k"]-273.15)
    T_network_HP_2=float(net.res_heat_consumer.at[HP_2.HC_inj_id,"t_from_k"]-273.15)
    error_1=abs(T_network_HP_1_loop-T_network_HP_1)
    error_2=abs(T_network_HP_2_loop-T_network_HP_2)
    print(error_1)
    print(error_2)
    T_network_HP_1_loop=T_network_HP_1
    T_network_HP_2_loop=T_network_HP_2
    iterations+=1


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 9.08e+05   | 0 %        | 2.02e+00   | 1.42e+06   | 4.03e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 4.56e+04   | 14 %       | 1.49e+00   | 1.97e+05   | 6.96e+03   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 7.04e+03   | 23 %       | 3.38e-01   | 1.64e+04   | 4.58e+02   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 1.73e+00   | 64 %       | 2.56e-06   | 1.58e+02   | 4.33e+00   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 1.70e-04   | 100 %      | 2.62e-12   | 1.41e-02   | 3.71e-04   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 3.45e-10   | 100 %      | 5.48e-15   | 4.94e-09   | 1.90e-08   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 6, Calculation time: 0.03 s, Iterations per second: 226.91

 iter  | residual   | pro

In [346]:
net.pipe

,name,from_junction,to_junction,std_type,length_km,inner_diameter_mm,outer_diameter_mm,k_mm,loss_coefficient,u_w_per_m2k,text_k,sections,in_service,type
0,Tubo_Ida_Plant_Nodo_1,0,2,None,0.0589,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
1,Tubo_RetornoPlant_Nodo_1,8,6,None,0.0589,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
2,nodo_1_nodo_2,2,3,None,0.0589,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
3,nodo_2_ret_nodo_1_ret,9,8,None,0.0589,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
4,Tubo_Ida_nodo_2_cons_1_ida,3,4,None,0.0376,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
5,Tubo_Retorno_nodo_2_cons_1_ret,10,9,None,0.0376,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
6,Tubo_Ida_nodo_2_cons_2_ida,3,5,None,0.0302,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
7,Tubo_Retorno_nodo_2_cons_2_ret,11,9,None,0.0302,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe


In [347]:
net.res_pipe

,v_mean_m_per_s,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,reynolds,lambda,dp_friction_loss_bar
0,-0.227743,3.000000,3.045543,314.742677,314.794053,314.742677,-3.990387,3.990387,-0.004025,53522.886442,0.451064,-4.554290e-02
1,-0.226943,2.454592,2.500000,304.960722,305.000000,304.960722,-3.990387,3.990387,-0.004010,44154.838411,0.451318,-4.540826e-02
2,-0.227748,3.045543,3.091087,314.794053,314.845492,314.794053,-3.990387,3.990387,-0.004025,53572.497890,0.451063,-4.554374e-02
3,-0.226940,2.409184,2.454592,304.891903,304.960722,304.921492,-3.990387,3.990387,-0.004010,44119.618922,0.451319,-4.540780e-02
4,0.000322,3.091087,3.091086,314.845492,297.066598,297.066598,0.005654,-0.005654,0.000006,63.827438,1.452572,1.874093e-07
5,0.000320,2.409184,2.409184,292.066598,304.891903,284.024991,0.005654,-0.005654,0.000006,42.217508,1.965827,2.524600e-07
6,-0.228074,3.091087,3.114505,314.845492,314.871854,314.845492,-3.996041,3.996041,-0.004030,53686.058236,0.451060,-2.341826e-02
7,-0.227257,2.385836,2.409184,304.871854,304.891903,304.871854,-3.996041,3.996041,-0.004016,44129.019944,0.451318,-2.334765e-02


In [348]:
net.heat_consumer

,name,from_junction,to_junction,qext_w,controlled_mdot_kg_per_s,deltat_k,treturn_k,in_service,type
0,HP_1_EXT,4,10,118.271242,NaN,5.0,NaN,True,heat_consumer
1,HP_1_INJ,10,4,-150000.000000,NaN,50.0,NaN,False,heat_consumer
2,HP_2_EXT,5,11,150000.000000,NaN,50.0,NaN,False,heat_consumer
3,HP_2_INJ,11,5,-167033.751951,NaN,-10.0,NaN,True,heat_consumer


In [349]:
net.res_heat_consumer


,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,3.091086,2.409184,297.066598,292.066598,292.066598,0.005654,-0.005654,0.000006,5.000000,118.271242
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.083402,-150000.000000
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.721854,150000.000000
3,2.385836,3.114505,304.871854,314.871854,314.871854,3.996041,-3.996041,0.004023,-10.000000,-167033.751951


In [350]:
net.res_circ_pump_pressure

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.850000,NaN
1,3.0,2.5,314.742677,305.0,305.0,3.990387,-3.990387,0.004017,9.742677,-162504.736526


In [351]:
# Segundo es no fijando un circpump 
import pandapipes as pp
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
import numpy as np
from tespy.tools import UserDefinedEquation


class Bidirectional_W_to_WHeatPump:

    def __init__( self,name,net,refrigerant,HC_ext_id,HC_inj_id):
        self.name = name
        self.refrigerant = refrigerant
        self.HC_ext_id=HC_ext_id
        self.HC_inj_id=HC_inj_id
        self.net=net
        self._build_tespy_Cycles()

    def _build_tespy_Cycles(self):
        def my_ude(ude):
            dt_DHN=ude.params['dt']
            return ude.conns[0].calc_T()+dt_DHN-ude.conns[1].calc_T()
        def my_ude_dependents(ude):
            c1, c2 = ude.conns
            return [c1.p,c1.h, c2.p,c2.h]
        self.nw_Cooling_net = Network()
        self.nw_Cooling_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Cooling_net_compressor = Compressor("compresor")
        self.Cooling_net_condenser = Condenser("condensador")
        self.Cooling_net_valve = Valve("valvula_expansion")
        self.Cooling_net_evaporator = HeatExchanger("evaporador")
        self.Cooling_net_cc=CycleCloser('CycleCloser')
        self.Cooling_net_source_consumer=Source("Source_Consumer ")
        self.Cooling_net_sink_consumer=Sink("Sink_consumer")
        self.Cooling_net_source_reseau=Source("Source_Reseau")
        self.Cooling_net_sink_reseau=Sink("Sink_Reseau")
        self.Cooling_net_c0=Connection(self.Cooling_net_valve, 'out1', self.Cooling_net_cc, 'in1', label='0')
        self.Cooling_net_c1 = Connection(self.Cooling_net_cc, 'out1', self.Cooling_net_evaporator, 'in2', label='1')
        self.Cooling_net_c2 = Connection(self.Cooling_net_evaporator, 'out2', self.Cooling_net_compressor, 'in1', label='2')
        self.Cooling_net_c3 = Connection(self.Cooling_net_compressor, 'out1', self.Cooling_net_condenser, 'in1', label='3')
        self.Cooling_net_c4 = Connection(self.Cooling_net_condenser, 'out1', self.Cooling_net_valve, 'in1', label='4')
        self.Cooling_net_c5=Connection(self.Cooling_net_condenser, 'out2',self.Cooling_net_sink_consumer, 'in1', label='5')
        self.Cooling_net_c6=Connection(self.Cooling_net_source_consumer, 'out1',self.Cooling_net_condenser , 'in2', label='6')
        self.Cooling_net_c7=Connection(self.Cooling_net_source_reseau, 'out1',self.Cooling_net_evaporator , 'in1', label='7')
        self.Cooling_net_c8=Connection(self.Cooling_net_evaporator, 'out1',self.Cooling_net_sink_reseau , 'in1', label='8')
        self.nw_Cooling_net.add_conns(self.Cooling_net_c0,  self.Cooling_net_c1,  self.Cooling_net_c2,  self.Cooling_net_c3,  self.Cooling_net_c4 , self.Cooling_net_c5,  self.Cooling_net_c6, self.Cooling_net_c7, self.Cooling_net_c8)
        self.Cooling_ude = UserDefinedEquation(
                    'my_ude', my_ude, my_ude_dependents, conns=[self.Cooling_net_c8, self.Cooling_net_c7],params={'dt': 5})
        self.nw_Cooling_net.add_ude(self.Cooling_ude)
        
        #Heating net 
        self.nw_Heating_net = Network()
        self.nw_Heating_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Heating_net_compressor = Compressor("compresor")
        self.Heating_net_condenser = Condenser("condensador")
        self.Heating_net_valve = Valve("valvula_expansion")
        self.Heating_net_evaporator = HeatExchanger("evaporador")
        self.Heating_net_cc=CycleCloser('CycleCloser')
        self.Heating_net_source_consumer=Source("Source_Consumer ")
        self.Heating_net_sink_consumer=Sink("Sink_consumer")
        self.Heating_net_source_reseau=Source("Source_Reseau")
        self.Heating_net_sink_reseau=Sink("Sink_Reseau")
        self.Heating_net_c0=Connection(self.Heating_net_valve, 'out1', self.Heating_net_cc, 'in1', label='0')
        self.Heating_net_c1 = Connection(self.Heating_net_cc, 'out1', self.Heating_net_evaporator, 'in2', label='1')
        self.Heating_net_c2 = Connection(self.Heating_net_evaporator, 'out2', self.Heating_net_compressor, 'in1', label='2')
        self.Heating_net_c3 = Connection(self.Heating_net_compressor, 'out1', self.Heating_net_condenser, 'in1', label='3')
        self.Heating_net_c4 = Connection(self.Heating_net_condenser, 'out1', self.Heating_net_valve, 'in1', label='4')
        self.Heating_net_c5 = Connection(self.Heating_net_evaporator, 'out1', self.Heating_net_sink_consumer, 'in1', label='5')
        self.Heating_net_c6 = Connection(self.Heating_net_source_consumer, 'out1', self.Heating_net_evaporator, 'in1', label='6')  
        self.Heating_net_c7 = Connection(self.Heating_net_source_reseau, 'out1', self.Heating_net_condenser, 'in2', label='7')
        self.Heating_net_c8 = Connection(self.Heating_net_condenser, 'out2', self.Heating_net_sink_reseau, 'in1', label='8')
        self.nw_Heating_net.add_conns(self.Heating_net_c0,  self.Heating_net_c1,  self.Heating_net_c2,  self.Heating_net_c3,  self.Heating_net_c4 , self.Heating_net_c5,  self.Heating_net_c6, self.Heating_net_c7, self.Heating_net_c8)
        self.Heating_ude = UserDefinedEquation(
                        'my_ude_4', my_ude, my_ude_dependents, conns=[self.Heating_net_c7, self.Heating_net_c8],params={'dt': 5})
        self.nw_Heating_net.add_ude(self.Heating_ude)
        
    def solve_cycle(self,mode,Q_consumer,eta_s,T_nework_in,T_cons,dt_DHN):
        #The pressure values for the consumer and district heating side in the heating pump are arbitrary values
        self.mode=mode
        T_cons_in = T_cons[0]
        T_cons_out = T_cons[1]
        T_DH_in = T_nework_in
        if self.mode=="COOLING_NET":
            self.Cooling_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
            self.Cooling_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5,Q=Q_consumer)
            self.Cooling_net_compressor.set_attr(eta_s=eta_s)
            self.Cooling_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Cooling_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Cooling_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Cooling_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Cooling_net._set_p_range([p_triple, p_high])
            self.nw_Cooling_net._set_h_range([h_min,h_max])
            self.Cooling_ude.params['dt'] = dt_DHN
            self.nw_Cooling_net.solve('design')        
        elif self.mode=="HEATING_NET":
            self.Heating_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5,Q=Q_consumer)
            self.Heating_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
            self.Heating_net_compressor.set_attr(eta_s=eta_s)
            self.Heating_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Heating_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Heating_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Heating_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Heating_net._set_p_range([p_triple, p_high])
            self.nw_Heating_net._set_h_range([h_min,h_max])
            self.Heating_ude.params['dt'] = dt_DHN
            self.nw_Heating_net.solve('design')
    def interaction_simulation(self,dT_water):
        if self.mode=="COOLING_NET":
            self.net.heat_consumer.at[self.HC_ext_id, "qext_w"] =abs( self.Cooling_net_evaporator.Q.val)
            self.net.heat_consumer.at[self.HC_ext_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False
            
        elif self.mode=="HEATING_NET":
            self.net.heat_consumer.at[self.HC_inj_id, "qext_w"] = self.Heating_net_condenser.Q.val
            self.net.heat_consumer.at[self.HC_inj_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False

        else: 
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False

    def get_main_results(self):
        if self.mode=="COOLING_NET":
            results = {
                "Q_consumer": abs(self.Cooling_net_condenser.Q.val),
                "Q_network": abs(self.Cooling_net_evaporator.Q.val),
                "W_compressor":self.Cooling_net_compressor.P.val ,
                "COP": abs(self.Cooling_net_condenser.Q.val)/self.Cooling_net_compressor.P.val 
            }
        elif self.mode=="HEATING_NET":
            results = {
                "Q_consumer": abs(self.Heating_net_evaporator.Q.val),
                "Q_network": abs(self.Heating_net_condenser.Q.val),
                "COP": abs(self.Heating_net_evaporator.Q.val)/self.Heating_net_compressor.P.val 
            }
        else:
            results = {}
        return results

    def get_results_solver(self):
        if self.mode=="COOLING_NET":
            self.nw_Cooling_net.print_results()
        elif self.mode=="HEATING_NET":
            self.nw_Heating_net.print_results()
        else:
            return None
net = pp.create_empty_network(fluid="water")
# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida") 
j_HP_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_Ida")
j_HP_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_Ida")
#Return
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret") 
j_HP_ret_1= pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_ret")
j_HP_ret_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_ret")

#Plant-Nodo
u=0.35 / (np.pi * 0.15)
#Planta 1
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u, text_k=273.2, name="Tubo_Ida_Plant_Nodo_1",k_mm=0.1*1000
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_RetornoPlant_Nodo_1",k_mm=0.1*1000)

#Nodo-nodo
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="nodo_1_nodo_2",k_mm=0.1*1000
)
pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="nodo_2_ret_nodo_1_ret",k_mm=0.1*1000
)

#Node HP
#Node_Cons1

pipe_ida_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_1,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Ida_nodo_2_cons_1_ida",k_mm=0.1*1000
)
pipe_retorno_3= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_1, to_junction=j_nodo_2_ret,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Retorno_nodo_2_cons_1_ret",k_mm=0.1*1000
)
#Node_Cons2
pipe_ida_4 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_2,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Ida_nodo_2_cons_2_ida",k_mm=0.1*1000
)
pipe_retorno_4= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_2, to_junction=j_nodo_2_ret,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Retorno_nodo_2_cons_2_ret",k_mm=0.1*1000
)

#Heat Pumps
#Heat pump: extraction
HP_1_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_1 ,
    to_junction=j_HP_ret_1,
    qext_w=150000,
    deltat_k=50,
    name="HP_1_EXT"
)
#Heat mump 1: Injection
HP_1_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_1,
    to_junction=j_HP_1,
    qext_w=-150000,
    deltat_k=50,
    name="HP_1_INJ"
)
#Heat pump: extraction
HP_2_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_2 ,
    to_junction=j_HP_ret_2,
    qext_w=150000,
    deltat_k=50,
    name="HP_2_EXT"
)
#Heat mump 1: Injection
HP_2_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_2,
    to_junction=j_HP_2,
    qext_w=150000,
    deltat_k=-50,
    name="HP_2_INJ"
)
#Plant: 

Plant_1_ida = pp.create_ext_grid(
    net, junction=j_fuente_ida, p_bar=3.0,
    type="p", name="Grid_ida",in_service=False
)
Plant_1_ret = pp.create_ext_grid(
    net, junction=j_fuente_ret, p_bar=2.5,t_k=305,   # = p_flow_bar - plift_bar, para conservar el mismo salto de presión
    type="pt", name="Grid_ret",in_service=False
)
Plant_1=pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3,plift_bar=0.5,t_flow_k=340 ,name='Grid')
T_network_inital_guess_1=35
T_network_inital_guess_2=50
HP_1=Bidirectional_W_to_WHeatPump(name="heat_pump_1",refrigerant="R134a",net=net,HC_ext_id=HP_1_Ext,HC_inj_id=HP_1_inj)
HP_2=Bidirectional_W_to_WHeatPump(name="heat_pump_2",refrigerant="R134a",net=net,HC_ext_id=HP_2_Ext,HC_inj_id=HP_2_inj)
T_network_HP_1=T_network_inital_guess_1
T_network_HP_2=T_network_inital_guess_2
T_network_HP_1_loop=T_network_inital_guess_1
T_network_HP_2_loop=T_network_inital_guess_2
tolerance=1E-6
error_1=10
error_2=10
iterations=0
while error_1>tolerance or error_2>tolerance:
    HP_1.solve_cycle(mode="COOLING_NET",Q_consumer=-150,eta_s=0.95,T_nework_in=T_network_HP_1,T_cons=[50,60],dt_DHN=5)
    HP_2.solve_cycle(mode="HEATING_NET",Q_consumer=-150000,eta_s=0.95,T_nework_in=T_network_HP_2,T_cons=[35,25],dt_DHN=10)
    print("iteration")
    HP_1.interaction_simulation(dT_water=5)
    HP_2.interaction_simulation(dT_water=-10)
    try:
        pp.pipeflow(net,mode="bidirectional") 
    except UserWarning: 
        net.circ_pump_pressure.at[Plant_1, "in_service"] =False
        net.ext_grid.at[Plant_1_ida, "in_service"] =True
        net.ext_grid.at[Plant_1_ret, "in_service"] =True 
        pp.pipeflow(net,mode="bidirectional")
        net.circ_pump_pressure.at[Plant_1, "in_service"] =True
        net.ext_grid.at[Plant_1_ida, "in_service"] =False
        net.ext_grid.at[Plant_1_ret, "in_service"] =False
    T_network_HP_1=float(net.res_heat_consumer.at[HP_1.HC_ext_id,"t_from_k"]-273.15)
    T_network_HP_2=float(net.res_heat_consumer.at[HP_2.HC_inj_id,"t_from_k"]-273.15)
    error_1=abs(T_network_HP_1_loop-T_network_HP_1)
    error_2=abs(T_network_HP_2_loop-T_network_HP_2)
    print(error_1)
    print(error_2)
    T_network_HP_1_loop=T_network_HP_1
    T_network_HP_2_loop=T_network_HP_2
    iterations+=1


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 9.08e+05   | 0 %        | 2.02e+00   | 1.42e+06   | 4.03e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 4.56e+04   | 14 %       | 1.49e+00   | 1.97e+05   | 6.96e+03   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 7.04e+03   | 23 %       | 3.38e-01   | 1.64e+04   | 4.58e+02   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 1.73e+00   | 64 %       | 2.56e-06   | 1.58e+02   | 4.33e+00   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 1.70e-04   | 100 %      | 2.62e-12   | 1.41e-02   | 3.71e-04   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 3.45e-10   | 100 %      | 5.48e-15   | 4.94e-09   | 1.90e-08   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 6, Calculation time: 0.03 s, Iterations per second: 239.82

 iter  | residual   | pro

In [352]:
net.res_pipe

,v_mean_m_per_s,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,reynolds,lambda,dp_friction_loss_bar
0,-0.227743,3.000000,3.045543,314.742677,314.794053,314.742677,-3.990387,3.990387,-0.004025,53522.886442,0.451064,-4.554290e-02
1,-0.226943,2.454592,2.500000,304.960722,305.000000,304.960722,-3.990387,3.990387,-0.004010,44154.838411,0.451318,-4.540826e-02
2,-0.227748,3.045543,3.091087,314.794053,314.845492,314.794053,-3.990387,3.990387,-0.004025,53572.497890,0.451063,-4.554374e-02
3,-0.226940,2.409184,2.454592,304.891903,304.960722,304.921492,-3.990387,3.990387,-0.004010,44119.618922,0.451319,-4.540780e-02
4,0.000322,3.091087,3.091086,314.845492,297.066598,297.066598,0.005654,-0.005654,0.000006,63.827438,1.452572,1.874093e-07
5,0.000320,2.409184,2.409184,292.066598,304.891903,284.024991,0.005654,-0.005654,0.000006,42.217508,1.965827,2.524600e-07
6,-0.228074,3.091087,3.114505,314.845492,314.871854,314.845492,-3.996041,3.996041,-0.004030,53686.058236,0.451060,-2.341826e-02
7,-0.227257,2.385836,2.409184,304.871854,304.891903,304.871854,-3.996041,3.996041,-0.004016,44129.019944,0.451318,-2.334765e-02
